In [ ]:
# %%
# Import required modules
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Import from your modules
from src.grotools import split_gro2residues
from src.iofile import read_gro
from src.site import Site

# Configure plotting
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)

print("✓ Imports successful")

In [ ]:
"""
## Step 1: Load GRO File and Create Site Objects
"""

# %%
# ==== CONFIGURE YOUR GRO FILE PATH HERE ====
gro_file = "/data/sgarg/pentacene/gb_pen_schellhammer/final_working_str/INITIAL_str/2_FINAL_EQ/a90.gro"

print(f"Loading: {gro_file}")
print("="*80)

# Read the .gro file
gro_data = read_gro(gro_file)
atom_lines = gro_data[2:-1]  # Skip header and footer

# Split into residues and create Site objects
residues = split_gro2residues(atom_lines)
sites = [Site(residue) for residue in residues]

print(f"Total residues/sites: {len(sites)}")
print(f"First residue ID: {sites[0].resid}")
print(f"Last residue ID: {sites[-1].resid}")

In [ ]:
"""
## Step 2: Calculate Center of Mass for Each Residue
"""

# %%
print("\n" + "="*80)
print("CALCULATING CENTERS OF MASS")
print("="*80)

# Extract COMs from all sites
coms = np.array([site.com for site in sites])

print(f"\nCOM array shape: {coms.shape}")
print(f"COM statistics:")
print(f"  X: min={coms[:, 0].min():.3f}, max={coms[:, 0].max():.3f}, mean={coms[:, 0].mean():.3f}")
print(f"  Y: min={coms[:, 1].min():.3f}, max={coms[:, 1].max():.3f}, mean={coms[:, 1].mean():.3f}")
print(f"  Z: min={coms[:, 2].min():.3f}, max={coms[:, 2].max():.3f}, mean={coms[:, 2].mean():.3f}")

# Show first 5 COMs
print(f"\nFirst 5 residue COMs:")
for i in range(min(5, len(sites))):
    com = sites[i].com
    print(f"  Residue {sites[i].resid}: [{com[0]:.4f}, {com[1]:.4f}, {com[2]:.4f}]")

In [ ]:
"""
## Step 3: Calculate Orientation Axes (Long, Short, Normal)
"""

# %%
print("\n" + "="*80)
print("CALCULATING ORIENTATION AXES VIA MASS-WEIGHTED PCA")
print("="*80)

# Storage for orientation data
orientation_data = []

for site in sites:
    # Compute mass-weighted PCA (3 components)
    axes, explained_var, rmat = site.mw_pca_axes(n_comps=3, rot='rmat')
    
    # axes[0] = long axis (largest variance)
    # axes[1] = short axis (medium variance)
    # axes[2] = normal (smallest variance, perpendicular to plane)
    
    long_axis = axes[0]
    short_axis = axes[1]
    normal_axis = axes[2]
    
    orientation_data.append({
        'resid': site.resid,
        'com_x': site.com[0],
        'com_y': site.com[1],
        'com_z': site.com[2],
        'long_x': long_axis[0],
        'long_y': long_axis[1],
        'long_z': long_axis[2],
        'short_x': short_axis[0],
        'short_y': short_axis[1],
        'short_z': short_axis[2],
        'normal_x': normal_axis[0],
        'normal_y': normal_axis[1],
        'normal_z': normal_axis[2],
        'var_long': explained_var[0],
        'var_short': explained_var[1],
        'var_ratio': explained_var[0] / explained_var[1] if explained_var[1] > 0 else 0
    })
    
# Create DataFrame
df = pd.DataFrame(orientation_data)

print(f"\nOrientation data extracted for {len(df)} residues")
print(f"\nDataFrame preview:")
print(df.head())

print(f"\nVariance statistics:")
print(f"  Long axis variance: mean={df['var_long'].mean():.4f}, std={df['var_long'].std():.4f}")
print(f"  Short axis variance: mean={df['var_short'].mean():.4f}, std={df['var_short'].std():.4f}")
print(f"  Variance ratio (long/short): mean={df['var_ratio'].mean():.4f}, std={df['var_ratio'].std():.4f}")


In [ ]:
"""
## Step 4: Distribution Analysis and Classification
"""

# %%
print("\n" + "="*80)
print("ANGULAR DISTRIBUTION ANALYSIS")
print("="*80)

# Calculate angles with global coordinate axes (X, Y, Z)
# For long axis
df['long_angle_x'] = np.degrees(np.arccos(np.abs(df['long_x'])))
df['long_angle_y'] = np.degrees(np.arccos(np.abs(df['long_y'])))
df['long_angle_z'] = np.degrees(np.arccos(np.abs(df['long_z'])))

# For normal axis
df['normal_angle_x'] = np.degrees(np.arccos(np.abs(df['normal_x'])))
df['normal_angle_y'] = np.degrees(np.arccos(np.abs(df['normal_y'])))
df['normal_angle_z'] = np.degrees(np.arccos(np.abs(df['normal_z'])))

print("\nLong axis alignment with global axes (degrees):")
print(f"  X-axis: mean={df['long_angle_x'].mean():.2f}°, std={df['long_angle_x'].std():.2f}°")
print(f"  Y-axis: mean={df['long_angle_y'].mean():.2f}°, std={df['long_angle_y'].std():.2f}°")
print(f"  Z-axis: mean={df['long_angle_z'].mean():.2f}°, std={df['long_angle_z'].std():.2f}°")

print("\nNormal axis alignment with global axes (degrees):")
print(f"  X-axis: mean={df['normal_angle_x'].mean():.2f}°, std={df['normal_angle_x'].std():.2f}°")
print(f"  Y-axis: mean={df['normal_angle_y'].mean():.2f}°, std={df['normal_angle_y'].std():.2f}°")
print(f"  Z-axis: mean={df['normal_angle_z'].mean():.2f}°, std={df['normal_angle_z'].std():.2f}°")

# %%
# Visualization 1: Long Axis Orientation Distribution
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Long Axis Orientation Distributions', fontsize=16, fontweight='bold')

# Long axis components
axes[0, 0].hist(df['long_x'], bins=30, edgecolor='black', alpha=0.7, color='steelblue')
axes[0, 0].set_xlabel('Long Axis X Component')
axes[0, 0].set_ylabel('Count')
axes[0, 0].axvline(df['long_x'].mean(), color='red', linestyle='--', label=f'Mean: {df["long_x"].mean():.3f}')
axes[0, 0].legend()

axes[0, 1].hist(df['long_y'], bins=30, edgecolor='black', alpha=0.7, color='steelblue')
axes[0, 1].set_xlabel('Long Axis Y Component')
axes[0, 1].set_ylabel('Count')
axes[0, 1].axvline(df['long_y'].mean(), color='red', linestyle='--', label=f'Mean: {df["long_y"].mean():.3f}')
axes[0, 1].legend()

axes[0, 2].hist(df['long_z'], bins=30, edgecolor='black', alpha=0.7, color='steelblue')
axes[0, 2].set_xlabel('Long Axis Z Component')
axes[0, 2].set_ylabel('Count')
axes[0, 2].axvline(df['long_z'].mean(), color='red', linestyle='--', label=f'Mean: {df["long_z"].mean():.3f}')
axes[0, 2].legend()

# Long axis angles
axes[1, 0].hist(df['long_angle_x'], bins=30, edgecolor='black', alpha=0.7, color='coral')
axes[1, 0].set_xlabel('Angle with X-axis (°)')
axes[1, 0].set_ylabel('Count')
axes[1, 0].axvline(df['long_angle_x'].mean(), color='red', linestyle='--', label=f'Mean: {df["long_angle_x"].mean():.1f}°')
axes[1, 0].legend()

axes[1, 1].hist(df['long_angle_y'], bins=30, edgecolor='black', alpha=0.7, color='coral')
axes[1, 1].set_xlabel('Angle with Y-axis (°)')
axes[1, 1].set_ylabel('Count')
axes[1, 1].axvline(df['long_angle_y'].mean(), color='red', linestyle='--', label=f'Mean: {df["long_angle_y"].mean():.1f}°')
axes[1, 1].legend()

axes[1, 2].hist(df['long_angle_z'], bins=30, edgecolor='black', alpha=0.7, color='coral')
axes[1, 2].set_xlabel('Angle with Z-axis (°)')
axes[1, 2].set_ylabel('Count')
axes[1, 2].axvline(df['long_angle_z'].mean(), color='red', linestyle='--', label=f'Mean: {df["long_angle_z"].mean():.1f}°')
axes[1, 2].legend()

plt.tight_layout()
plt.show()


In [ ]:
# Define classification criteria
# Example: classify by dominant orientation axis

def classify_long_axis_alignment(row, threshold_deg=30):
    """Classify residue by which global axis its long axis is most aligned with."""
    angles = {
        'X': row['long_angle_x'],
        'Y': row['long_angle_y'],
        'Z': row['long_angle_z']
    }
    min_axis = min(angles, key=angles.get)
    min_angle = angles[min_axis]
    
    if min_angle < threshold_deg:
        return f'aligned_with_{min_axis}'
    else:
        return 'mixed'

def classify_normal_axis_alignment(row, threshold_deg=30):
    """Classify residue by which global axis its normal is most aligned with."""
    angles = {
        'X': row['normal_angle_x'],
        'Y': row['normal_angle_y'],
        'Z': row['normal_angle_z']
    }
    min_axis = min(angles, key=angles.get)
    min_angle = angles[min_axis]
    
    if min_angle < threshold_deg:
        return f'normal_to_{min_axis}'
    else:
        return 'mixed'

# Apply classifications
df['long_axis_class'] = df.apply(classify_long_axis_alignment, axis=1)
df['normal_axis_class'] = df.apply(classify_normal_axis_alignment, axis=1)

# Summary statistics
print("\nLong Axis Classification:")
print(df['long_axis_class'].value_counts())
print(f"\nPercentages:")
print(df['long_axis_class'].value_counts(normalize=True) * 100)

print("\n" + "-"*80)
print("Normal Axis Classification:")
print(df['normal_axis_class'].value_counts())
print(f"\nPercentages:")
print(df['normal_axis_class'].value_counts(normalize=True) * 100)

# %%
# Visualization 4: Classification Pie Charts
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Orientation Classification Summary', fontsize=16, fontweight='bold')

# Long axis classification
long_counts = df['long_axis_class'].value_counts()
axes[0].pie(long_counts.values, labels=long_counts.index, autopct='%1.1f%%', startangle=90)
axes[0].set_title('Long Axis Alignment')

# Normal axis classification
normal_counts = df['normal_axis_class'].value_counts()
axes[1].pie(normal_counts.values, labels=normal_counts.index, autopct='%1.1f%%', startangle=90)
axes[1].set_title('Normal Axis Alignment')

plt.tight_layout()
plt.show()

In [ ]:
# %%
"""
## Step 5: 3D Views Colored by Orientation Classes
### 5.1 Long-axis alignment
"""
import py3Dmol

# Define a simple color map for long-axis classes
long_color_map = {
    'aligned_with_X': 'red',
    'aligned_with_Y': 'green',
    'aligned_with_Z': 'blue',
    'mixed': 'gray'
}

print("\n" + "="*80)
print("3D VIEW: Residues colored by LONG-AXIS alignment")
print("="*80)

view_long = py3Dmol.view(width=800, height=600)

for row in df.itertuples(index=False):
    cls = row.long_axis_class
    color = long_color_map.get(cls, 'gray')

    sphere = {
        'center': {
            'x': float(row.com_x),
            'y': float(row.com_y),
            'z': float(row.com_z),
        },
        'radius': 0.3,
        'color': color,
        'alpha': 0.9,
    }
    view_long.addSphere(sphere)

view_long.setBackgroundColor('0xeeeeee')
view_long.zoomTo()
view_long.show()

print("Color legend (long axis):")
for k, v in long_color_map.items():
    print(f"  {k:15s} -> {v}")
